In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 8.5 Final Model Selection and Deployment
- Recommended model: Survey-Enhanced XGBoost
- Deploy to hold-out students -> risk scores -> practical outreach threshold -> risk bands
- **Data note:** `Deploy_Survey_Data.csv`/`Deploy_Data_Other.csv` are not yet available locally — pending confirmation (see CHANGELOG).

## Setup

In [ ]:
import numpy as np
import pandas as pd
import pickle

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Load Deploy Data and Best Model
Uses the two files still pending confirmation with Keval — `Deploy_Survey_Data.csv` (features) and `Deploy_Data_Other.csv` (SID/NAME/LAST_NAME identifiers).

In [ ]:
df_deploy = pd.read_csv('../data/Deploy_Survey_Data.csv')
df_deploy_names = pd.read_csv('../data/Deploy_Data_Other.csv')

survey_xgb_model = pickle.load(open('../models/Survey_xgb_model.pkl', 'rb'))
print("Model and deploy data loaded.")

## Generate Risk Scores

In [ ]:
holdout_prob = survey_xgb_model.predict_proba(df_deploy)[:, 1]
holdout_pred_default = (holdout_prob >= 0.50).astype(int)

holdout_scores = df_deploy.copy()
holdout_scores['departure_risk_score'] = holdout_prob
holdout_scores['predicted_departed_default_050'] = holdout_pred_default

holdout_scores[['departure_risk_score', 'predicted_departed_default_050']].describe().round(4)

## Capacity-Based Outreach Threshold
A plain 0.50 cutoff may not match advising capacity. Flag the top 20% by risk instead — a practical, resource-aware threshold.

In [ ]:
capacity_share = 0.20
capacity_threshold = float(np.quantile(holdout_prob, 1 - capacity_share))
holdout_scores['flag_top_20pct_capacity'] = (holdout_scores['departure_risk_score'] >= capacity_threshold).astype(int)

print(f"Capacity threshold for top {capacity_share:.0%}: {capacity_threshold:.4f}")
print("Number flagged:", int(holdout_scores['flag_top_20pct_capacity'].sum()))

## Risk Bands for Stakeholders

In [ ]:
def assign_risk_band(score):
    if score >= np.quantile(holdout_prob, 0.80):
        return 'Priority outreach'
    elif score >= np.quantile(holdout_prob, 0.50):
        return 'Monitor/support'
    return 'Routine support'

holdout_scores['support_band'] = holdout_scores['departure_risk_score'].apply(assign_risk_band)
holdout_scores['support_band'].value_counts()

## Advisor-Facing Outreach List

In [ ]:
df_deploy_risk = pd.concat([df_deploy_names[['SID', 'NAME', 'LAST_NAME']], holdout_scores], axis=1)

recommended_context_cols = ['SID', 'NAME', 'LAST_NAME', 'departure_risk_score', 'support_band',
    'flag_top_20pct_capacity', 'HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2',
    'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']

advisor_outreach_list = df_deploy_risk[recommended_context_cols].sort_values('departure_risk_score', ascending=False).reset_index(drop=True)
advisor_outreach_list.head(20).round(4)

## Summary
- Survey-Enhanced XGBoost deployed as a **decision-support tool**, not a decision-making system.
- Risk bands (not raw probabilities) make results actionable for advisors.
- **Known issue, not fixed here:** the outreach list joins names to scores by row position (`pd.concat(axis=1)`), not by SID — matches the lecture's current (flagged, unresolved) approach. If the two files aren't in identical row order, this silently mismatches students to the wrong risk score. See CHANGELOG for the recommended fix, pending approval.

**Module 8 Complete!**